In [ ]:
from huggingface_hub import login
login()

In [ ]:
using_colab = True

In [ ]:
if using_colab:
    import sys, os, pathlib, re
    import torch, torchvision

    print("PyTorch version:", torch.__version__)
    print("Torchvision version:", torchvision.__version__)
    print("CUDA is available:", torch.cuda.is_available())

    !{sys.executable} -m pip install -q decord ultralytics

    repo_dir = "/content/sam3"
    if os.path.exists(repo_dir):
        !rm -rf {repo_dir}
    !git clone -q https://github.com/facebookresearch/sam3.git {repo_dir}

    pyproject = pathlib.Path(repo_dir) / "pyproject.toml"
    text = pyproject.read_text()
    text = re.sub(r'"numpy==[^"]+",?', "", text)
    pyproject.write_text(text)

    %cd {repo_dir}
    !{sys.executable} -m pip install -q -e .
    %cd /content

    if "sam3" in sys.modules:
        del sys.modules["sam3"]
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)

In [ ]:
# Mount the drive
from google.colab import drive
drive.mount('/content/drive/')
DRIVE_PATH = "/content/drive/MyDrive" #update the path with the location of zipfile

In [ ]:
# Unzip the dataset
DATASET_DIR_NAME = "duckietown_dataset"
DATASET_ZIP_NAME = f"{DATASET_DIR_NAME}.zip"
DATASET_DIR_PATH = os.path.join("/content", DATASET_DIR_NAME)
TRAIN_DIR = "train"
VALIDATION_DIR = "val"
IMAGES_DIR = "images"
LABELS_DIR = "labels"


def show_info(base_path: str):
  for l1 in [TRAIN_DIR, VALIDATION_DIR]:
    for l2 in [IMAGES_DIR, LABELS_DIR]:
      p = os.path.join(base_path, l1, l2)
      print(f"#Files in {l1}/{l2}: {len(os.listdir(p))}")


def unzip_dataset():
  # check zipped file
  zip_path = os.path.join(DRIVE_PATH, DATASET_ZIP_NAME)
  assert os.path.exists(zip_path), f"No zipped dataset found at {zip_path}! Abort!"

  # unzip the data
  print("Unpacking zipped data...")
  shutil.unpack_archive(zip_path, DATASET_DIR_PATH)
  print(f"Zipped dataset unpacked to {DATASET_DIR_PATH}")

  # show some info
  show_info(DATASET_DIR_PATH)


unzip_dataset()

In [ ]:
DATASET_DIR = Path(DATASET_DIR_PATH)
TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "val"
CLASSES_YAML = DATASET_DIR / "classes.yaml"

In [ ]:
%%writefile /content/duckietown_dataset/classes.yaml

# train and val data as 1) directory: path/images/, 2) file: path/images.txt, or 3) list: [path1/images/, path2/images/]
train:  /content/duckietown_dataset/train
val:    /content/duckietown_dataset/val

# class names
names:
  0: 'yellow rubber duck' # you can try adding more classes if you'd like

In [ ]:
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

In [ ]:
!wget https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz -P /content/sam3/assets/

In [ ]:
def load_classes(classes_yaml):
  with open(classes_yaml, "r") as f:
    cfg = yaml.safe_load(f)
  items = sorted(cfg["names"].items(), key=lambda kv: int(kv[0]))
  return [name for _, name in items]


def xyxy_to_yolo_line(bbox, w, h, class_id):
  x1, y1, x2, y2 = bbox
  cx = (x1 + x2) / 2.0 / w
  cy = (y1 + y2) / 2.0 / h
  bw = (x2 - x1) / w
  bh = (y2 - y1) / h
  return f"{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"


class Sam3AutoLabel:
  def __init__(
    self,
    train_dir,
    val_dir,
    dataset_dir,
    classes_yaml,
    bpe_path="/content/sam3/assets/bpe_simple_vocab_16e6.txt.gz",
    confidence_threshold=0.3,
    device=None,
  ):
    self.train_dir = train_dir
    self.val_dir = val_dir
    self.dataset_dir = dataset_dir
    self.classes_yaml = classes_yaml

    if device is None:
      device = "cuda" if torch.cuda.is_available() else "cpu"
    self.device = device

    self.classes = load_classes(self.classes_yaml)

    self.model = build_sam3_image_model(bpe_path=str(bpe_path))
    self.model.to(device)
    self.model.eval()

    self.processor = Sam3Processor(
      self.model,
      confidence_threshold=confidence_threshold,
      device=device,
    )

  def _detect(self, img_path: Path):
    image = Image.open(img_path).convert("RGB")
    w, h = image.size

    with torch.no_grad():
      if self.device == "cuda":
        ctx = torch.autocast("cuda", dtype=torch.bfloat16)
      else:
        from contextlib import nullcontext
        ctx = nullcontext()

      with ctx:
        st = self.processor.set_image(image)

        dets = []
        for class_id, prompt in enumerate(self.classes):
          out = self.processor.set_text_prompt(state=st, prompt=prompt)
          boxes = out.get("boxes")
          scores = out.get("scores")
          if boxes is None or scores is None:
            continue

          for b, s in zip(boxes, scores):
            s = float(s)
            if s < self.processor.confidence_threshold:
              continue
            dets.append(
              {
                  "bbox": b.tolist(),
                  "score": s,
                  "class_id": int(class_id),
              }
            )
                    
    return dets, (w, h)

  def _label_split(self, split: str, max_images=None):
    import random  # kan ook bovenaan je script

    split_dir = self.train_dir if split == "train" else self.val_dir
    img_dir = split_dir / "images"
    lbl_dir = split_dir / "labels"

    img_paths = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        img_paths.extend(img_dir.rglob(ext))

    # 🔥 random subset i.p.v. eerste N
    if max_images is not None:
        img_paths = random.sample(img_paths, min(max_images, len(img_paths)))

    for img_path in tqdm(img_paths, desc=f"Labeling {split}"):
        dets, (w, h) = self._detect(img_path)

        yolo_lines = []
        for d in dets:
            yolo_lines.append(
                xyxy_to_yolo_line(d["bbox"], w, h, d["class_id"])
            )

        label_path = lbl_dir / f"{img_path.stem}.txt"
        if yolo_lines:
            label_path.write_text("\n".join(yolo_lines))
        else:
            if label_path.exists():
                label_path.unlink()

  def run(self, max_images=None):
    self._label_split("train", max_images=max_images)
    self._label_split("val", max_images=max_images)

In [ ]:
def visualize_labels_for_image(img_path: Path, labels_dir: Path, class_names=None, save_to: Path | None = None):
  img_path = Path(img_path)
  label_path = labels_dir / (img_path.stem + ".txt")

  if not label_path.exists():
    return

  image = Image.open(img_path).convert("RGB")
  w, h = image.size

  with open(label_path, "r") as f:
    lines = [ln.strip() for ln in f.readlines() if ln.strip()]

  fig, ax = plt.subplots(figsize=(6, 6))
  ax.imshow(image)
  ax.axis("off")

  for line in lines:
    parts = line.split()
    if len(parts) != 5:
      continue
    class_id = int(parts[0])
    cx, cy, bw, bh = map(float, parts[1:])

    box_w = bw * w
    box_h = bh * h
    x1 = cx * w - box_w / 2
    y1 = cy * h - box_h / 2

    rect = plt.Rectangle(
      (x1, y1),
      box_w,
      box_h,
      fill=False,
      linewidth=2,
    )
    ax.add_patch(rect)

    label = str(class_id)
    if class_names is not None and 0 <= class_id < len(class_names):
      label = class_names[class_id]

    ax.text(
      x1,
      y1 - 2,
      label,
      fontsize=10,
      bbox=dict(facecolor="black", alpha=0.5),
      color="white",
    )

  if save_to is not None:
    save_to = Path(save_to)
    save_to.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_to, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
  else:
    plt.show()

In [ ]:
autolabel = Sam3AutoLabel(
    train_dir=TRAIN_DIR,
    val_dir=VAL_DIR,
    dataset_dir=DATASET_DIR,
    classes_yaml=CLASSES_YAML,
    confidence_threshold=0.3,
)

autolabel.run()

In [ ]:
train_images = TRAIN_DIR / "images"
train_labels = TRAIN_DIR / "labels"

label = next(train_labels.glob("*.txt"))

img_name = label.stem + ".png"
img = train_images / img_name

if not img.exists():
    raise FileNotFoundError(f"Image not found for label {label.name}")

visualize_labels_for_image(
    img_path=img,
    labels_dir=train_labels,
    class_names=load_classes(CLASSES_YAML),
)